# 02 · Run experiments
Everything is resumable: finished `(model, task)` pairs are skipped. Run groups separately; check `results/` after each. Same tasks, caps, and
context length for every method (`configs/protocol.yaml`). Use `tmux`/`nohup` for long runs (the cells below just shell out to `scripts/run_model.py`).

In [ ]:
import os, sys, json, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, yaml
pd.set_option("display.width", 160, "display.max_columns", 30)

In [ ]:
TIER = "core"                       # smoke | core | full
SAVE_PRED = "--save-predictions"    # needed for notebook 04 (mechanism probes); costs disk, not time
def run(models, extra=""):
    !python scripts/run_model.py --tier $TIER --models {" ".join(models)} $SAVE_PRED {extra}

### A · Statistical references (minutes)

In [ ]:
run(["seasonal_naive", "naive", "seasonal_avg", "drift"])

### B · Zero-shot foundation models (GPU, ~minutes per model per task)
Run one at a time so a crash in one wrapper doesn't block the rest.

In [ ]:
for m in ["chronos_t5_small", "timesfm", "moirai", "sundial", "moment_zs"]:
    run([m])

### C · LLM-as-forecaster, zero-shot (slow: autoregressive text generation)
Capped by protocol to 100 instances/task and horizon ≤ 96 (`per_model_overrides`). Expect many `unsupported` tasks — that is reported, not hidden.

In [ ]:
run(["llmtime"])

### D · Supervised, trained per task (GPU; most expensive block)
Start with `--seeds 0`. For the paper, rerun with `--seeds 0 1 2` and average (`aggregate` handles `@s<seed>` suffixes).

In [ ]:
for m in ["patchtst", "informer", "autoformer", "fedformer"]:
    run([m], "--seeds 0")

In [ ]:
for m in ["gpt4ts", "time_llm"]:
    run([m], "--seeds 0")

### E · Tan et al. ablations + random-GPT-2 control (answers *"do LLM weights help?"*)

In [ ]:
for m in ["tan_wo_llm", "tan_llm2attn", "tan_llm2trsf", "gpt2_scratch"]:
    run([m], "--seeds 0")

### Progress / failure audit

In [ ]:
from tscompare import aggregate as ag
df = ag.load_results("results")
display(df.pivot_table(index="model", columns="status", values="task", aggfunc="count", fill_value=0))
display(df[df.status == "failed"][["model", "task", "error"]].head(20))